# Prompt Engineering for Agentic AI

## Overview

This notebook explores prompt engineering techniques for building effective agentic AI systems, including dynamic prompt chains and chain-of-thought reasoning.

## Learning Objectives

- Master fundamental principles of prompt engineering
- Build dynamic prompt chains that adapt based on context
- Implement chain-of-thought reasoning for complex problems
- Apply few-shot learning to guide agent behavior
- Create conditional prompts for different complexity levels

## Exam Objectives: 2.1, 2.6
## Estimated Time: 75-90 minutes

## 🎯 Exam Focus

This notebook covers concepts that appear frequently on the NCP-AAI exam:

**High-Priority Topics:**
- ⭐⭐⭐ Error handling patterns (retry logic, circuit breakers)
- ⭐⭐⭐ Prompt engineering techniques
- ⭐⭐ Tool integration best practices
- ⭐⭐ Streaming responses

**Common Exam Scenarios:**
- Implementing retry logic for API failures
- Designing prompts for complex tasks
- Handling tool execution errors

**Key Concepts to Master:**
- Exponential backoff for retries
- Circuit breaker pattern
- Dynamic prompt chains
- Graceful degradation

## Setup: Import Dependencies

In [ ]:
# Import core libraries for prompt engineering
import os
import sys
from typing import Dict, List, Any, Optional
from datetime import datetime

# LangChain imports for prompt templates and chains
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain, SequentialChain

# Verify LangChain is available
try:
    import langchain
    print(f"✅ LangChain version: {langchain.__version__}")
except ImportError as e:
    # Handle missing dependency gracefully
    print(f"❌ LangChain not available: {e}")
    print("   Install with: pip install langchain")

# Check for NVIDIA AI Endpoints (optional for this notebook)
try:
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    print("✅ NVIDIA AI Endpoints available")
    NVIDIA_AVAILABLE = True
except ImportError:
    print("⚠️  NVIDIA AI Endpoints not available (optional for exercises)")
    print("   Install with: pip install langchain-nvidia-ai-endpoints")
    NVIDIA_AVAILABLE = False

print("\n🎯 Setup complete!")

## Theory: Prompt Engineering Fundamentals

### What is Prompt Engineering?

**Prompt engineering** is the art and science of crafting inputs that elicit desired behaviors from language models. For agentic AI systems, effective prompts are critical for:

- **Guiding behavior**: Defining agent roles and capabilities
- **Ensuring consistency**: Maintaining coherent responses across interactions
- **Enabling reasoning**: Encouraging step-by-step problem solving
- **Controlling output**: Specifying format and constraints

### Five Key Principles

1. **Clarity** - Be explicit about what you want
   - ❌ "Tell me about agents"
   - ✅ "Explain the three main agent architectures (reactive, deliberative, hybrid) with use cases for each"

2. **Context** - Provide relevant background information
   - Include user history, current state, available tools
   - Example: "User is a premium customer with 3 previous support tickets"

3. **Structure** - Use consistent formatting
   - Organize prompts with clear sections
   - Use delimiters (###, ---, etc.) to separate components

4. **Examples** - Show desired output format (few-shot learning)
   - Provide 2-5 examples of input-output pairs
   - Helps model understand expected behavior

5. **Constraints** - Specify limitations and requirements
   - Length limits, format requirements, prohibited content
   - Example: "Respond in 3 sentences or less"

### Agent Prompt Template Structure

```
ROLE: You are {agent_role}
PURPOSE: {agent_purpose}

CAPABILITIES:
- {capability_1}
- {capability_2}

GUIDELINES:
- {guideline_1}
- {guideline_2}

CONTEXT: {current_context}

USER REQUEST: {user_input}

RESPONSE:
```

## Implementation: Basic Prompt Templates

Let's build prompt templates for different agent types.

In [ ]:
# Create a customer support agent prompt template
support_agent_template = PromptTemplate(
    input_variables=["customer_name", "customer_tier", "issue", "history"],
    template="""You are a professional customer support agent for TechCorp.

CUSTOMER INFORMATION:
- Name: {customer_name}
- Tier: {customer_tier}
- Previous Interactions: {history}

CURRENT ISSUE:
{issue}

GUIDELINES:
1. Acknowledge the customer's concern with empathy
2. Ask clarifying questions if needed
3. Provide step-by-step solutions
4. Escalate to human agent if customer is frustrated or issue is complex
5. Maintain professional and friendly tone

RESPONSE:
"""
)

# Test the template with sample data
formatted_prompt = support_agent_template.format(
    customer_name="Alice Johnson",
    customer_tier="Premium",
    issue="My order #12345 hasn't arrived yet. It was supposed to be here yesterday.",
    history="Previous order delivered successfully 2 weeks ago"
)

print("=== Customer Support Agent Prompt ===")
print(formatted_prompt)
print("\n" + "="*50 + "\n")

# Create a research assistant prompt template
research_agent_template = PromptTemplate(
    input_variables=["topic", "depth", "sources"],
    template="""You are a research assistant specializing in technical topics.

RESEARCH REQUEST:
Topic: {topic}
Depth: {depth}
Required Sources: {sources}

INSTRUCTIONS:
1. Break down the topic into key subtopics
2. Identify the most important concepts to cover
3. Synthesize information from multiple perspectives
4. Provide citations for key claims
5. Highlight areas of uncertainty or debate

RESEARCH FINDINGS:
"""
)

# Test research template
research_prompt = research_agent_template.format(
    topic="Retrieval-Augmented Generation (RAG) architectures",
    depth="Comprehensive",
    sources="Academic papers, technical blogs, documentation"
)

print("=== Research Assistant Prompt ===")
print(research_prompt)

## Implementation: Dynamic Prompt Chains

Dynamic prompt chains adapt based on intermediate results, enabling complex multi-step reasoning.

In [ ]:
# Simulated LLM for demonstration (replace with real LLM in production)
class MockLLM:
    """Mock LLM for demonstration purposes."""
    
    def __init__(self):
        # Predefined responses for demonstration
        self.responses = {
            "extract": "Entities: quantum computing, latest developments, applications\nIntent: Information gathering",
            "search": "1. Recent breakthroughs in quantum computing 2024\n2. Quantum computing practical applications\n3. Quantum computing vs classical computing advantages",
            "synthesize": "Quantum computing has seen significant developments in 2024, including improved error correction and new applications in drug discovery and cryptography."
        }
    
    def invoke(self, prompt: str) -> str:
        """Simulate LLM invocation."""
        # Simple keyword matching for demo
        if "Extract" in prompt or "entities" in prompt.lower():
            return self.responses["extract"]
        elif "search queries" in prompt.lower():
            return self.responses["search"]
        else:
            return self.responses["synthesize"]

# Create mock LLM instance
mock_llm = MockLLM()

# Build a sequential chain for research workflow
# Step 1: Extract key information from user query
extract_chain = LLMChain(
    llm=mock_llm,
    prompt=PromptTemplate(
        input_variables=["user_query"],
        template="""Extract the key entities and intent from this query:

Query: {user_query}

Entities and Intent:
"""
    ),
    output_key="entities"
)

# Step 2: Generate search queries based on extracted entities
search_chain = LLMChain(
    llm=mock_llm,
    prompt=PromptTemplate(
        input_variables=["entities"],
        template="""Based on these entities and intent:
{entities}

Generate 3 specific search queries to find relevant information:
"""
    ),
    output_key="search_queries"
)

# Step 3: Synthesize final response
synthesis_chain = LLMChain(
    llm=mock_llm,
    prompt=PromptTemplate(
        input_variables=["user_query", "search_queries"],
        template="""Original query: {user_query}

Search queries generated: {search_queries}

Provide a comprehensive answer to the original query:
"""
    ),
    output_key="final_answer"
)

# Combine into sequential chain
research_workflow = SequentialChain(
    chains=[extract_chain, search_chain, synthesis_chain],
    input_variables=["user_query"],
    output_variables=["entities", "search_queries", "final_answer"],
    verbose=True
)

# Execute the chain
print("=== Dynamic Prompt Chain Execution ===")
result = research_workflow({"user_query": "What are the latest developments in quantum computing?"})

print("\n=== Chain Results ===")
print(f"\nExtracted Entities:\n{result['entities']}")
print(f"\nGenerated Search Queries:\n{result['search_queries']}")
print(f"\nFinal Answer:\n{result['final_answer']}")

## Implementation: Conditional Prompt Chains

Conditional chains select different prompts based on input characteristics.

In [ ]:
# Implementation of conditional prompt chain
class ConditionalPromptChain:
    """Chain that selects prompts based on query complexity."""
    
    def __init__(self, llm):
        """
        Initialize conditional chain with LLM.
        
        Args:
            llm: Language model to use for generation
        """
        self.llm = llm
        
        # Define prompt templates for different complexity levels
        self.simple_prompt = PromptTemplate(
            input_variables=["query"],
            template="""Provide a brief, concise answer:

Question: {query}

Answer (2-3 sentences):
"""
        )
        
        self.detailed_prompt = PromptTemplate(
            input_variables=["query"],
            template="""Provide a comprehensive answer with examples:

Question: {query}

Answer:
1. Overview
2. Key concepts
3. Practical examples
4. Summary
"""
        )
        
        self.technical_prompt = PromptTemplate(
            input_variables=["query"],
            template="""Provide a technical explanation with code examples:

Question: {query}

Technical Answer:
1. Technical overview
2. Implementation details
3. Code examples
4. Best practices
"""
        )
    
    def assess_complexity(self, query: str) -> str:
        """
        Assess query complexity to select appropriate prompt.
        
        Args:
            query: User query to assess
        
        Returns:
            Complexity level: 'simple', 'detailed', or 'technical'
        """
        query_lower = query.lower()
        
        # Check for technical indicators
        technical_keywords = ['implement', 'code', 'algorithm', 'function', 'class', 'api']
        if any(keyword in query_lower for keyword in technical_keywords):
            return 'technical'
        
        # Check for detailed explanation indicators
        detailed_keywords = ['explain', 'how', 'why', 'compare', 'analyze', 'describe']
        if any(keyword in query_lower for keyword in detailed_keywords):
            return 'detailed'
        
        # Default to simple
        return 'simple'
    
    def execute(self, query: str, complexity: Optional[str] = None) -> str:
        """
        Execute chain with appropriate prompt based on complexity.
        
        Args:
            query: User query
            complexity: Optional explicit complexity level
        
        Returns:
            Generated response
        """
        # Auto-detect complexity if not provided
        if complexity is None:
            complexity = self.assess_complexity(query)
        
        print(f"Detected complexity: {complexity}")
        
        # Select appropriate prompt
        if complexity == 'simple':
            prompt = self.simple_prompt
        elif complexity == 'detailed':
            prompt = self.detailed_prompt
        else:
            prompt = self.technical_prompt
        
        # Create and execute chain
        chain = LLMChain(llm=self.llm, prompt=prompt)
        result = chain.run(query=query)
        return result

# Test conditional chain
conditional_chain = ConditionalPromptChain(mock_llm)

# Test with different query types
test_queries = [
    "What is RAG?",  # Simple
    "Explain how RAG systems work and their benefits",  # Detailed
    "How do I implement a RAG pipeline with LangChain?"  # Technical
]

print("=== Conditional Prompt Chain Testing ===")
for query in test_queries:
    print(f"\nQuery: {query}")
    response = conditional_chain.execute(query)
    print(f"Response: {response}")
    print("-" * 60)

## References

### Course Materials

**Course Notes:** [Module 2: Agent Development](../../course-notes/module-02-agent-development.md)

### Practice Questions

**Exam Questions:** [Practice Questions](../../exam-questions/domain-02-development.md)

### Hands-On Labs

- [Lab 1: Basic RAG Agent](../../labs/lab-01-basic-rag-agent/README.md)
- [Lab 2: Multi-Agent Research](../../labs/lab-02-multi-agent-research/README.md)

### Additional Resources

- [NVIDIA NeMo Documentation](https://docs.nvidia.com/nemo/)
- [LangChain Documentation](https://python.langchain.com/)
- [NVIDIA AI Endpoints](https://build.nvidia.com/)
